In [13]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import rateslib as rl
import QuantLib as ql

import datetime
import pytz

NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
LDN_tz = pytz.timezone("Europe/London") 
UTC_tz = pytz.timezone("UTC") 

import sys
sys.path.append("../")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [14]:
from MDP.IRSwaptions.IRSwaptionMDP import IRSwaptionMDP
from Query.Base.query_resolution import resolve_query
from Query.IRSwaptions import IRSwaptionQuery, IRSwaptionStructure, IRSwaptionValue


In [15]:
# mdp = IRSwaptionMDP(
#     source="GSQUANT-QL",
#     curve_source="ERIS_EOD_LIVE-QL_BASIC",
# )

mdp = IRSwaptionMDP(
    source="MONKEYCUBE-QL",
    curve_source="ERIS_EOD_LIVE-QL_BASIC",
    data_dir=r'C:\Users\chris\clee\ARBS\MDP\IRSwaptions\MONKEYCUBE\YCMONKEY_USD_VOL_CUBE_GAMMA_MIX'
)

In [16]:
as_of = datetime.date(2026, 3, 10)
ctx = mdp.get_pricer(
    {
        "curve_name": "USD-SOFR-1D",
        "timestamp": as_of,
        "ignore_cache": True,
    }
)
ctx

IRSwaptionMarketContext(curve_name='USD-SOFR-1D', as_of_date=datetime.date(2026, 3, 10), curve=QLIRSwapCurve(_ql_curve_id='USD-SOFR-1D', _ql_curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x0000018DB201C7B0> >, _ql_curve_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x0000018DB201C770> >, _meta_data={'timestamp': datetime.date(2026, 3, 10)}), curve_handle=<QuantLib.QuantLib.YieldTermStructureHandle; proxy of <Swig Object of type 'Handle< YieldTermStructure > *' at 0x0000018DB201C7B0> >, swap_index=<QuantLib.QuantLib.Sofr; proxy of <Swig Object of type 'ext::shared_ptr< Sofr > *' at 0x0000018DB201C770> >, vol_handle=<QuantLib.QuantLib.SwaptionVolatilityStructureHandle; proxy of <Swig Object of type 'Handle< SwaptionVolatilityStructure > *' at 0x0000018DB201D330> >, pricing_engine=<QuantLib.QuantLib.BachelierSwaptionEngine; proxy of <Swig Object of type 'ext::shared_

In [17]:
q = IRSwaptionQuery(
	shorthand="1y1y",
	structure=IRSwaptionStructure.PAYER,
    strike="ATMF+100",
)

q_eff = resolve_query(q, timestamp=as_of, pricer_or_curve=ctx)
package, weights = q_eff.resolve_package(pricer_or_curve=ctx)
vmap = q_eff.build_value_map(pricer_or_curve=ctx, package=package, risk_weights=weights)
float(vmap.apply(IRSwaptionValue.NVOL)), float(vmap.apply(IRSwaptionValue.FWD_PREM))

(89.61876329241137, 5.995192849622616)

In [18]:
q = IRSwaptionQuery(
	shorthand="1y1y",
	structure=IRSwaptionStructure.RECEIVER,
    strike="ATMF-100",
)

q_eff = resolve_query(q, timestamp=as_of, pricer_or_curve=ctx)
package, weights = q_eff.resolve_package(pricer_or_curve=ctx)
vmap = q_eff.build_value_map(pricer_or_curve=ctx, package=package, risk_weights=weights)
float(vmap.apply(IRSwaptionValue.NVOL)), float(vmap.apply(IRSwaptionValue.FWD_PREM))

(105.61966527650542, 9.756155382926154)